In [2]:
import os 
%pwd

'd:\\Text Summarizer\\research'

In [3]:
os.chdir('../')
%pwd

'd:\\Text Summarizer'

In [4]:
from dataclasses import dataclass
from pathlib import Path

In [15]:
@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [16]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [17]:
class ConfigurationManager:
    def __init__(self, config_file_path: Path = CONFIG_FILE_PATH, params_file_path: Path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self)-> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])
        data_transformation_config = DataTransformationConfig(root_dir=config.root_dir, 
                                                              data_path=config.data_path,
                                                              tokenizer_name=config.tokenizer_name)
        return data_transformation_config

In [18]:
import os 
from src.textSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [19]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)


    def convert_examples_to_features(self,example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'] , max_length = 1024, truncation = True )
        
        target_encodings = self.tokenizer(example_batch['summary'], max_length = 128, truncation = True )
            
        return {
            'input_ids' : input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))

In [20]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-04-08 01:49:38,704]: INFO: common: yaml file: config\config.yaml loaded sucessfully
[2026-04-08 01:49:38,705]: INFO: common: yaml file: params.yaml loaded sucessfully
[2026-04-08 01:49:38,707]: INFO: common: Created Directory at : artifacts
[2026-04-08 01:49:38,708]: INFO: common: Created Directory at : artifacts/data_transformation
[2026-04-08 01:49:39,120]: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-08 01:49:39,175]: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"
[2026-04-08 01:49:39,445]: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
[2026-04-08 01:49:39,478]: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 46689.63 examples/s]
